# Barcelona Traffic Feature Engineering — 2025/2026
**Available data:** January 2025 – June 2026 (8 monthly files; May 2026 absent from source)  
**Goal:** Apply the same aggregation pipeline as `traffic_aggregate_2017-2018.ipynb` to the recent dataset,
then compare `idTram` coverage against the 2017–2018 features and the full noise-streets reference layer.

| Time block | Hours |
|---|---|
| **Day (Ld)** | 07:00–20:59 |
| **Evening (Le)** | 21:00–22:59 |
| **Night (Ln)** | 23:00–06:59 |

In [16]:
import pandas as pd
import numpy as np
import geopandas as gpd
import glob
import os

## 1. File Discovery
Select all files in the `2025/` and `2026/` subdirectories.  
No further date filtering is needed — every file in those folders is within scope.

In [17]:
TRAFFIC_DIR = os.path.normpath(os.path.join('..', '..', 'layers', 'traffic'))

all_files = sorted(glob.glob(os.path.join(TRAFFIC_DIR, '**', '*_TRAMS_TRAMS.csv'), recursive=True))

def is_recent(filepath):
    year = int(os.path.basename(filepath).split('_')[0])
    return year in (2025, 2026)

target_files = [f for f in all_files if is_recent(f)]

print(f'Found {len(target_files)} files:')
for f in target_files:
    print(f'  {os.path.basename(f)}')

Found 8 files:
  2025_01_Gener_TRAMS_TRAMS.csv
  2025_02_Febrer_TRAMS_TRAMS.csv
  2025_03_Marc_TRAMS_TRAMS.csv
  2026_01_Gener_TRAMS_TRAMS.csv
  2026_02_Febrer_TRAMS_TRAMS.csv
  2026_03_Marc_TRAMS_TRAMS.csv
  2026_04_Abril_TRAMS_TRAMS.csv
  2026_06_Juny_TRAMS_TRAMS.csv


## 2. Aggregation Pipeline

Identical to the 2017–2018 notebook:
- Read one file at a time (memory-safe)
- Mask `estatActual == 0` to NaN for valid-only statistics
- Carry intermediate sums (`sum_estat`, `sum_sq_estat`) for a mathematically correct pooled std across months

`traffic_volatility = sqrt( (Σx² − (Σx)²/n) / (n−1) )`

In [18]:
def assign_time_block(hour_series):
    conditions = [
        (hour_series >= 7) & (hour_series <= 20),
        (hour_series >= 21) & (hour_series <= 22),
    ]
    return np.select(conditions, ['Day', 'Evening'], default='Night')

In [19]:
def process_month(filepath):
    df = pd.read_csv(
        filepath,
        dtype={'idTram': 'int32', 'data': 'int64', 'estatActual': 'int8', 'estatPrevist': 'int8'}
    )

    df['ts'] = pd.to_datetime(df['data'].astype(str), format='%Y%m%d%H%M%S')
    if df.empty:
        return pd.DataFrame()

    df['time_block']  = assign_time_block(df['ts'].dt.hour)
    df['estat_valid'] = df['estatActual'].where(df['estatActual'] != 0).astype('float32')

    agg = df.groupby(['idTram', 'time_block'], observed=True).agg(
        total_rows      = ('estatActual', 'size'),
        valid_rows      = ('estatActual', lambda x: (x != 0).sum()),
        fluid_count     = ('estatActual', lambda x: x.isin([1, 2]).sum()),
        congested_count = ('estatActual', lambda x: (x == 5).sum()),
        sum_estat       = ('estat_valid', 'sum'),
        sum_sq_estat    = ('estat_valid', lambda x: x.pow(2).sum()),
    ).reset_index()

    return agg

## 3. Batch Processing Loop

In [20]:
monthly_summaries = []

for fp in target_files:
    name = os.path.basename(fp)
    print(f'Processing {name} ...', end=' ', flush=True)
    summary = process_month(fp)
    if not summary.empty:
        monthly_summaries.append(summary)
        print(f'{len(summary):,} segment×block rows')
    else:
        print('EMPTY – skipped')

monthly_df = pd.concat(monthly_summaries, ignore_index=True)
print(f'\nStacked summaries: {monthly_df.shape[0]:,} rows × {monthly_df.shape[1]} cols')

Processing 2025_01_Gener_TRAMS_TRAMS.csv ... 1,596 segment×block rows
Processing 2025_02_Febrer_TRAMS_TRAMS.csv ... 1,596 segment×block rows
Processing 2025_03_Marc_TRAMS_TRAMS.csv ... 1,596 segment×block rows
Processing 2026_01_Gener_TRAMS_TRAMS.csv ... 1,596 segment×block rows
Processing 2026_02_Febrer_TRAMS_TRAMS.csv ... 1,596 segment×block rows
Processing 2026_03_Marc_TRAMS_TRAMS.csv ... 1,596 segment×block rows
Processing 2026_04_Abril_TRAMS_TRAMS.csv ... 1,596 segment×block rows
Processing 2026_06_Juny_TRAMS_TRAMS.csv ... 1,596 segment×block rows

Stacked summaries: 12,768 rows × 8 cols


## 4. Final Consolidation

In [21]:
global_agg = (
    monthly_df
    .groupby(['idTram', 'time_block'], as_index=False)
    [['total_rows', 'valid_rows', 'fluid_count', 'congested_count', 'sum_estat', 'sum_sq_estat']]
    .sum()
)

n = global_agg['valid_rows']

global_agg['pct_fluid']         = global_agg['fluid_count']     / n
global_agg['pct_congested']     = global_agg['congested_count'] / n
global_agg['sensor_uptime_pct'] = n / global_agg['total_rows']

numerator = global_agg['sum_sq_estat'] - global_agg['sum_estat'].pow(2) / n
global_agg['traffic_volatility'] = np.sqrt(numerator / (n - 1))

features = global_agg[[
    'idTram', 'time_block',
    'pct_fluid', 'pct_congested', 'traffic_volatility', 'sensor_uptime_pct'
]].copy()

print(f'Long-form table:      {features.shape[0]:,} rows')
print(f'Unique idTram values: {features["idTram"].nunique():,}')
features.head(9)

Long-form table:      1,596 rows
Unique idTram values: 532


,idTram,time_block,pct_fluid,pct_congested,traffic_volatility,sensor_uptime_pct
0,1,Day,0.665799,0.011384,0.749073,0.983210
1,1,Evening,0.981982,0.000000,0.536276,0.958963
2,1,Night,0.974101,0.000000,0.477986,0.413914
3,2,Day,0.936763,0.000000,0.435287,0.964099
4,2,Evening,0.974912,0.000000,0.544473,0.740389
5,2,Night,0.989238,0.000000,0.486038,0.365894
6,3,Day,0.975688,0.000250,0.219185,0.952608
7,3,Evening,0.994072,0.000000,0.392806,0.728726
8,3,Night,0.998835,0.000000,0.505328,0.281667


## 5. Pivot to Wide Format

In [22]:
METRICS      = ['pct_fluid', 'pct_congested', 'traffic_volatility', 'sensor_uptime_pct']
BLOCK_PREFIX = {'Day': 'day', 'Evening': 'evening', 'Night': 'night'}

wide = features.pivot_table(index='idTram', columns='time_block', values=METRICS)
wide.columns = [f'{BLOCK_PREFIX[block]}_{metric}' for metric, block in wide.columns]
wide = wide.reset_index()

ordered_cols = ['idTram']
for prefix in ['day', 'evening', 'night']:
    ordered_cols += [f'{prefix}_{m}' for m in METRICS]

wide = wide[ordered_cols]

print(f'Wide-format table: {wide.shape[0]:,} rows × {wide.shape[1]} columns')
wide.head()

Wide-format table: 532 rows × 13 columns


,idTram,day_pct_fluid,day_pct_congested,day_traffic_volatility,day_sensor_uptime_pct,evening_pct_fluid,evening_pct_congested,evening_traffic_volatility,evening_sensor_uptime_pct,night_pct_fluid,night_pct_congested,night_traffic_volatility,night_sensor_uptime_pct
0,1,0.665799,0.011384,0.749073,0.983210,0.981982,0.000000,0.536276,0.958963,0.974101,0.000000,0.477986,0.413914
1,2,0.936763,0.000000,0.435287,0.964099,0.974912,0.000000,0.544473,0.740389,0.989238,0.000000,0.486038,0.365894
2,3,0.975688,0.000250,0.219185,0.952608,0.994072,0.000000,0.392806,0.728726,0.998835,0.000000,0.505328,0.281667
3,4,0.675377,0.005863,0.746935,0.954632,0.901533,0.003538,0.539271,0.732613,0.995038,0.000292,0.515500,0.374754
4,5,0.893824,0.015563,0.542513,0.967849,0.971578,0.005220,0.610995,0.744708,0.978523,0.000377,0.566933,0.290308


## 6. Coverage Comparison
How many unique `idTram` segments does the recent dataset cover compared to:
- the 2017–2018 feature file (2 004 segments)
- the full `BCN_noise_streets` reference layer

The noise-streets layer is the ground truth: it lists every street segment for which a noise level was modelled.

In [23]:
# --- Load reference noise-streets layer ---
NOISE_PATH = os.path.normpath(os.path.join('..', '..', 'layers', 'BCN_noise_streets.gpkg'))
noise_gdf  = gpd.read_file(NOISE_PATH)

print('Noise-streets columns:', list(noise_gdf.columns))
print(f'Total features in gpkg: {len(noise_gdf):,}')
noise_gdf.head(3)

Noise-streets columns: ['TRAM', 'TOTAL_D', 'TOTAL_E', 'TOTAL_N', 'TOTAL_DEN', 'TRANSIT_D', 'TRANSIT_E', 'TRANSIT_N', 'TRANSIT_DEN', 'GI_TR_D', 'GI_TR_E', 'GI_TR_N', 'GI_TR_DEN', 'FFCC_D', 'FFCC_E', 'FFCC_N', 'FFCC_DEN', 'INDUST_D', 'INDUST_E', 'INDUST_N', 'INDUST_DEN', 'VIANANTS_D', 'VIANANTS_E', 'OCI_N', 'PATIS_D', 'PATIS_E', 'geometry_type', 'start', 'end', 'geometry']
Total features in gpkg: 15,115


,TRAM,TOTAL_D,TOTAL_E,TOTAL_N,TOTAL_DEN,TRANSIT_D,TRANSIT_E,TRANSIT_N,TRANSIT_DEN,GI_TR_D,...,INDUST_DEN,VIANANTS_D,VIANANTS_E,OCI_N,PATIS_D,PATIS_E,geometry_type,start,end,geometry
0,T04719W,70 - 75 dB(A),65 - 70 dB(A),60 - 65 dB(A),70 - 75 dB(A),70 - 75 dB(A),65 - 70 dB(A),60 - 65 dB(A),70 - 75 dB(A),< 40 dB(A),...,< 40 dB(A),< 40 dB(A),< 40 dB(A),< 40 dB(A),< 40 dB(A),< 40 dB(A),Line,430230,430182,"MULTILINESTRING ((430229.789 4586585.199, 4301..."
1,T19941Z,45 - 50 dB(A),45 - 50 dB(A),< 40 dB(A),45 - 50 dB(A),40 - 45 dB(A),40 - 45 dB(A),< 40 dB(A),45 - 50 dB(A),< 40 dB(A),...,< 40 dB(A),45 - 50 dB(A),45 - 50 dB(A),< 40 dB(A),< 40 dB(A),< 40 dB(A),Line,432929,432995,"MULTILINESTRING ((432928.898 4584019.988, 4329..."
2,T18111R,55 - 60 dB(A),55 - 60 dB(A),50 - 55 dB(A),55 - 60 dB(A),55 - 60 dB(A),55 - 60 dB(A),50 - 55 dB(A),55 - 60 dB(A),40 - 45 dB(A),...,< 40 dB(A),< 40 dB(A),< 40 dB(A),< 40 dB(A),< 40 dB(A),< 40 dB(A),Line,429953,430025,"MULTILINESTRING ((429953.263 4588161.441, 4299..."


In [24]:
# --- Load 2017-2018 features (if available) ---
OLD_PATH = os.path.normpath(os.path.join('..', '..', 'data', 'processed', 'barcelona_traffic_features_2017_2018.csv'))

ids_old      = set()
ids_recent   = set(wide['idTram'].astype(int))

if os.path.exists(OLD_PATH):
    old = pd.read_csv(OLD_PATH, usecols=['idTram'])
    ids_old = set(old['idTram'].astype(int))
    print(f'2017-2018 unique idTram: {len(ids_old):,}')
else:
    print('2017-2018 feature file not found – skipping.')

print(f'2025-2026 unique idTram: {len(ids_recent):,}')

2017-2018 unique idTram: 534
2025-2026 unique idTram: 532


In [25]:
# --- Identify the idTram column in the noise layer ---
# Try common column name variants
tram_col = None
for candidate in ['idTram', 'IDTRAM', 'id_tram', 'TRAM_ID', 'tramId', 'id']:
    if candidate in noise_gdf.columns:
        tram_col = candidate
        break

if tram_col:
    ids_noise = set(noise_gdf[tram_col].dropna().astype(int))
    print(f'Noise-streets unique {tram_col}: {len(ids_noise):,}')

    in_recent_not_old  = ids_recent - ids_old   if ids_old else set()
    in_old_not_recent  = ids_old   - ids_recent if ids_old else set()
    in_noise_not_recent= ids_noise - ids_recent
    in_recent_not_noise= ids_recent - ids_noise

    print()
    print('=== Coverage Report ===')
    if ids_old:
        print(f'  Segments in 2025-26 but NOT in 2017-18:  {len(in_recent_not_old):,}  <- new streets!')
        print(f'  Segments in 2017-18 but NOT in 2025-26:  {len(in_old_not_recent):,}  <- sensors gone offline')
    print(f'  Segments in noise layer not in 2025-26:  {len(in_noise_not_recent):,}')
    print(f'  Segments in 2025-26 not in noise layer:  {len(in_recent_not_noise):,}')
    print(f'  Overlap (2025-26 ∩ noise layer):         {len(ids_recent & ids_noise):,}')
else:
    print('Could not find an idTram column in the noise layer.')
    print('Please inspect the columns above and set tram_col manually.')

Could not find an idTram column in the noise layer.
Please inspect the columns above and set tram_col manually.


In [27]:
# --- New segment IDs found in 2025-26 that were absent in 2017-18 ---
if ids_old and in_recent_not_old:
    new_ids_df = wide[wide['idTram'].isin(in_recent_not_old)][['idTram']].copy()
    print(f'New idTram values (in 2025-26, absent in 2017-18): {len(new_ids_df):,}')
    new_ids_df.sort_values('idTram').head(20)

NameError: name 'in_recent_not_old' is not defined

## 7. Export

In [ ]:
OUT_PATH = os.path.normpath(os.path.join('..', '..', 'data', 'processed', 'barcelona_traffic_features_2025_2026.csv'))

wide.to_csv(OUT_PATH, index=False)

size_kb = os.path.getsize(OUT_PATH) / 1024
print(f'Saved:  {OUT_PATH}')
print(f'Size:   {size_kb:.1f} KB')
print(f'Shape:  {wide.shape[0]:,} rows × {wide.shape[1]} columns')

Saved:  ..\..\data\barcelona_traffic_features_2025_2026.csv
Size:   93.5 KB
Shape:  532 rows × 13 columns
